# Fortaleza de la economía familiar
## Un análisis exploratorio sobre territorio, ingresos y tamaño del hogar

**Encuesta Permanente de Hogares (EPH), tercer trimestre de 2025**

Esta notebook adapta a un formato reproducible un trabajo realizado para **Análisis de Datos I** de la Licenciatura en Análisis y Gestión de Datos de la Universidad Nacional de San Luis.

**Versión notebook:** Chris Moreno  
**Trabajo original (Grupo 1):** Rocío Cacciamano, Federico Granero, Chris Moreno, Julián Reinoso y Jeremías Blejman.

### Preguntas de análisis

1. ¿Cómo se distribuye el ingreso per cápita familiar (IPCF) en la muestra?
2. ¿Qué diferencias territoriales aparecen entre aglomerados urbanos?
3. Entre los hogares que no logran sostenerse con ingresos laborales, ¿qué estrategias financieras aparecen?
4. ¿Qué relación existe entre el tamaño del hogar y el IPCF?

### Fuente y alcance

Los datos provienen de la **Encuesta Permanente de Hogares (EPH) del INDEC, tercer trimestre de 2025**. El archivo incluido en este repositorio contiene seis variables seleccionadas de la base de hogares.

> **Nota metodológica:** esta versión de la base no incluye la variable de ponderación (`PONDERA`). Por lo tanto, los resultados que siguen describen **la muestra disponible** y no deben interpretarse como estimaciones poblacionales ponderadas para toda la Argentina.

Fuentes:
- INDEC, bases de microdatos EPH: https://www.indec.gob.ar/indec/web/Institucional-Indec-BasesDeDatos
- Diseño de registros EPH T3 2025: https://www.indec.gob.ar/ftp/cuadros/menusuperior/eph/EPH_registro_3T2025.pdf

### Resultados destacados

- **15.860 hogares** en la base seleccionada.
- El **21,2%** declara no poder vivir exclusivamente de sus ingresos laborales.
- Dentro de ese grupo, **23,9%** utilizó ahorros y **9,6%** recurrió a préstamos bancarios/financieros.
- El IPCF tiene una **media de $546.504** y una **mediana de $375.000**, con un coeficiente de variación cercano al **151,9%**.
- El tamaño del hogar se concentra en **2–3 integrantes** (media = 2,83; mediana = 3; moda = 2).
- La relación entre tamaño del hogar e IPCF es negativa (**rho de Spearman = −0,437**), pero la regresión lineal simple explica sólo **9,2%** de la variación del IPCF.
- En la comparación no ponderada por aglomerado, **Ushuaia–Río Grande** presenta el IPCF medio más alto y **Gran Resistencia** el más bajo.

Estos resultados son descriptivos de la muestra y deben leerse junto con la limitación principal del archivo: la selección de variables no incluye `PONDERA`.

In [ ]:
from pathlib import Path
from io import BytesIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display, Markdown

LOCAL_DATA = Path("data/eph_hogares_t3_2025.csv")
DRIVE_FILE_ID = "1iFhZ9kuLQKp7kw9dHoTMf02zO6krapjH"
DRIVE_DOWNLOAD = f"https://drive.google.com/uc?export=download&id={DRIVE_FILE_ID}"

if LOCAL_DATA.exists():
    df = pd.read_csv(LOCAL_DATA)
else:
    import requests

    response = requests.get(DRIVE_DOWNLOAD, timeout=60)
    response.raise_for_status()
    raw = pd.read_excel(BytesIO(response.content), sheet_name="Datos", header=None)

    # El archivo original contiene dos filas de encabezados.
    df = raw.iloc[2:, :6].copy()
    df.columns = ["AGLOMERADO", "IX_TOT", "IPCF", "V1", "V13", "V15"]

    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["AGLOMERADO", "IX_TOT", "IPCF"]).reset_index(drop=True)

# Los códigos 9 en V1, V13 y V15 representan NS/NR.
for col in ["V1", "V13", "V15"]:
    df[col] = df[col].replace(9, np.nan)

df.head()

## 1. Estructura de los datos

In [ ]:
diccionario = pd.DataFrame({
    "Variable": ["AGLOMERADO", "IX_TOT", "IPCF", "V1", "V13", "V15"],
    "Descripción": [
        "Código del aglomerado urbano",
        "Cantidad de integrantes del hogar",
        "Ingreso per cápita familiar ($)",
        "¿Han vivido de lo que ganan en el trabajo?",
        "¿Han tenido que gastar lo que tenían ahorrado?",
        "¿Han pedido préstamos a bancos/financieras?"
    ],
    "Tipo": [
        "Cualitativa nominal",
        "Cuantitativa discreta",
        "Cuantitativa continua",
        "Cualitativa binaria",
        "Cualitativa binaria",
        "Cualitativa binaria"
    ]
})

display(diccionario)

resumen_calidad = pd.DataFrame({
    "filas": [len(df)],
    "columnas": [df.shape[1]],
    "IPCF_igual_0": [(df["IPCF"] == 0).sum()],
    "IPCF_igual_0_pct": [(df["IPCF"] == 0).mean() * 100],
    "faltantes_V1": [df["V1"].isna().sum()],
    "faltantes_V13": [df["V13"].isna().sum()],
    "faltantes_V15": [df["V15"].isna().sum()]
})
display(resumen_calidad.round(2))

El valor `IPCF = 0` se conserva en el análisis descriptivo porque forma parte de la base seleccionada. No se interpreta automáticamente como “pobreza extrema”: puede corresponder a hogares sin ingresos declarados y también estar afectado por subdeclaración u otras limitaciones de medición.

## 2. Sostén económico y estrategias financieras

In [ ]:
def porcentaje_si(serie):
    serie = serie.dropna()
    return (serie.eq(1).mean() * 100)

frecuencias = pd.DataFrame({
    "Indicador": [
        "Vive de ingresos del trabajo (V1 = Sí)",
        "Usa ahorros (V13 = Sí)",
        "Solicita préstamos (V15 = Sí)"
    ],
    "Porcentaje": [
        porcentaje_si(df["V1"]),
        porcentaje_si(df["V13"]),
        porcentaje_si(df["V15"])
    ]
})

display(frecuencias.round(1))

In [ ]:
vulnerables = df[df["V1"] == 2].copy()

estrategias = pd.Series({
    "Uso de ahorros": porcentaje_si(vulnerables["V13"]),
    "Préstamos bancarios": porcentaje_si(vulnerables["V15"])
})

plt.figure(figsize=(7, 4))
estrategias.plot(kind="bar")
plt.ylabel("Hogares (%)")
plt.xlabel("")
plt.title("Estrategias de sostenimiento entre hogares que no viven del ingreso laboral")
plt.xticks(rotation=0)
plt.ylim(0, max(estrategias) * 1.25)
for i, value in enumerate(estrategias):
    plt.text(i, value + 0.5, f"{value:.1f}%", ha="center")
plt.tight_layout()
plt.show()

display(Markdown(
    f"En la muestra, **{(df['V1'].eq(2).mean()*100):.1f}%** de los hogares declara que no logra vivir "
    f"de lo que obtiene por el trabajo. Dentro de ese subgrupo, **{estrategias['Uso de ahorros']:.1f}%** "
    f"recurrió a ahorros y **{estrategias['Préstamos bancarios']:.1f}%** solicitó préstamos bancarios o financieros."
))

## 3. Tamaño de los hogares

In [ ]:
tam = df["IX_TOT"]

resumen_tam = pd.Series({
    "n": tam.count(),
    "media": tam.mean(),
    "mediana": tam.median(),
    "moda": tam.mode().iloc[0],
    "desviación estándar": tam.std(),
    "varianza": tam.var(),
    "coeficiente de variación (%)": tam.std() / tam.mean() * 100,
    "mínimo": tam.min(),
    "máximo": tam.max()
}).to_frame("valor")

display(resumen_tam.round(2))

In [ ]:
conteo_tam = df["IX_TOT"].value_counts().sort_index()

plt.figure(figsize=(9, 4.5))
conteo_tam.plot(kind="bar")
plt.xlabel("Cantidad de integrantes")
plt.ylabel("Cantidad de hogares")
plt.title("Distribución del tamaño de los hogares")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Distribución del ingreso per cápita familiar

In [ ]:
ipcf = df["IPCF"]

resumen_ipcf = pd.Series({
    "n": ipcf.count(),
    "media": ipcf.mean(),
    "mediana": ipcf.median(),
    "desviación estándar": ipcf.std(),
    "coeficiente de variación (%)": ipcf.std() / ipcf.mean() * 100,
    "mínimo": ipcf.min(),
    "máximo": ipcf.max(),
    "percentil 75": ipcf.quantile(.75),
    "percentil 90": ipcf.quantile(.90),
    "percentil 99": ipcf.quantile(.99)
}).to_frame("valor")

display(resumen_ipcf.round(2))

In [ ]:
# Para que la forma de la distribución sea legible, el gráfico se limita visualmente
# al percentil 99. Los casos superiores NO se eliminan de la base.
limite_visual = df["IPCF"].quantile(.99)

plt.figure(figsize=(8, 4.5))
plt.hist(df.loc[df["IPCF"] <= limite_visual, "IPCF"], bins=50)
plt.xlabel("IPCF ($)")
plt.ylabel("Frecuencia")
plt.title("Distribución del IPCF (visualización hasta el percentil 99)")
plt.tight_layout()
plt.show()

display(Markdown(
    f"La media del IPCF es **${df['IPCF'].mean():,.0f}**, mientras que la mediana es "
    f"**${df['IPCF'].median():,.0f}**. La distancia entre ambas medidas, junto con un coeficiente "
    f"de variación de **{df['IPCF'].std()/df['IPCF'].mean()*100:.1f}%**, muestra una distribución "
    "fuertemente asimétrica hacia la derecha."
))

## 5. Diferencias territoriales

In [ ]:
AGLOMERADOS = {
    2: "Gran La Plata", 3: "Bahía Blanca-Cerri", 4: "Gran Rosario",
    5: "Gran Santa Fe", 6: "Gran Paraná", 7: "Posadas",
    8: "Gran Resistencia", 9: "Comodoro Rivadavia-Rada Tilly",
    10: "Gran Mendoza", 12: "Corrientes", 13: "Gran Córdoba",
    14: "Concordia", 15: "Formosa", 17: "Neuquén-Plottier",
    18: "Santiago del Estero-La Banda", 19: "Jujuy-Palpalá",
    20: "Río Gallegos", 22: "Gran Catamarca", 23: "Gran Salta",
    25: "La Rioja", 26: "Gran San Luis", 27: "Gran San Juan",
    29: "Gran Tucumán-Tafí Viejo", 30: "Santa Rosa-Toay",
    31: "Ushuaia-Río Grande", 32: "Ciudad Autónoma de Buenos Aires",
    33: "Partidos del Gran Buenos Aires", 34: "Mar del Plata",
    36: "Río Cuarto", 38: "San Nicolás-Villa Constitución",
    91: "Rawson-Trelew", 93: "Viedma-Carmen de Patagones"
}

df["aglomerado_nombre"] = df["AGLOMERADO"].map(AGLOMERADOS)

territorio = (
    df.groupby("aglomerado_nombre")["IPCF"]
      .agg(casos="size", media="mean", mediana="median")
      .sort_values("media", ascending=False)
)

display(Markdown("**Cinco aglomerados con mayor IPCF medio en la muestra**"))
display(territorio.head(5).round(0))

display(Markdown("**Cinco aglomerados con menor IPCF medio en la muestra**"))
display(territorio.tail(5).sort_values("media").round(0))

In [ ]:
territorio_mediana = territorio.sort_values("mediana")

plt.figure(figsize=(9, 10))
plt.barh(territorio_mediana.index, territorio_mediana["mediana"])
plt.xlabel("Mediana de IPCF ($)")
plt.ylabel("")
plt.title("Mediana de IPCF por aglomerado urbano")
plt.tight_layout()
plt.show()

La comparación territorial debe interpretarse con cautela: se presentan estadísticas **no ponderadas**. Se utiliza la mediana como complemento de la media porque el IPCF presenta una asimetría muy marcada y valores extremos.

## 6. Tamaño del hogar e ingreso per cápita

In [ ]:
# Para el análisis bivariado se excluyen IPCF=0 y el valor extremo de $50.000.000.
reg = df[(df["IPCF"] > 0) & (df["IPCF"] < 50_000_000)].copy()

rho, p_spearman = stats.spearmanr(reg["IX_TOT"], reg["IPCF"])
modelo = stats.linregress(reg["IX_TOT"], reg["IPCF"])

resumen_modelo = pd.DataFrame({
    "Indicador": [
        "Casos analizados",
        "Spearman rho",
        "p (Spearman)",
        "Pendiente OLS",
        "Intercepto OLS",
        "R²"
    ],
    "Valor": [
        len(reg),
        rho,
        p_spearman,
        modelo.slope,
        modelo.intercept,
        modelo.rvalue**2
    ]
})
display(resumen_modelo)

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(reg["IX_TOT"], reg["IPCF"], alpha=0.12, s=12)

x_line = np.linspace(reg["IX_TOT"].min(), reg["IX_TOT"].max(), 100)
y_line = modelo.intercept + modelo.slope * x_line
plt.plot(x_line, y_line)

plt.xlabel("Integrantes del hogar")
plt.ylabel("IPCF ($)")
plt.title("Relación entre tamaño del hogar e IPCF")
plt.tight_layout()
plt.show()

display(Markdown(
    f"La asociación monotónica es negativa y moderada (**rho de Spearman = {rho:.3f}**): "
    "a mayor cantidad de integrantes, tiende a disminuir el ingreso per cápita. "
    f"Sin embargo, el ajuste lineal explica solamente **{modelo.rvalue**2*100:.1f}%** de la variación del IPCF, "
    "por lo que el tamaño del hogar es insuficiente como predictor único."
))

### Diagnóstico del modelo lineal

In [ ]:
reg["predicho"] = modelo.intercept + modelo.slope * reg["IX_TOT"]
reg["residuo"] = reg["IPCF"] - reg["predicho"]

plt.figure(figsize=(7, 4.5))
plt.scatter(reg["predicho"], reg["residuo"], alpha=0.12, s=12)
plt.axhline(0)
plt.xlabel("Valor predicho")
plt.ylabel("Residuo")
plt.title("Residuos vs. valores predichos")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
stats.probplot(reg["residuo"], dist="norm", plot=plt)
plt.title("Q-Q plot de los residuos")
plt.tight_layout()
plt.show()

Los gráficos de diagnóstico muestran que los residuos no se comportan como ruido aproximadamente normal y homocedástico. Por eso, la recta puede ser útil como resumen descriptivo de tendencia, pero **no constituye un modelo satisfactorio para predecir el IPCF**.

## 7. Conclusiones

In [ ]:
pct_no_vive = df["V1"].eq(2).mean() * 100
pct_ahorros_vuln = porcentaje_si(vulnerables["V13"])
pct_prest_vuln = porcentaje_si(vulnerables["V15"])
media_tam = df["IX_TOT"].mean()
mediana_tam = df["IX_TOT"].median()

display(Markdown(f'''
- El IPCF presenta **una fuerte asimetría positiva**: la media (${df["IPCF"].mean():,.0f}) es considerablemente mayor que la mediana (${df["IPCF"].median():,.0f}).
- **{pct_no_vive:.1f}%** de los hogares de la muestra declara no poder vivir exclusivamente de sus ingresos laborales.
- Dentro de ese grupo, el **uso de ahorros ({pct_ahorros_vuln:.1f}%)** aparece con mayor frecuencia que la **solicitud de préstamos ({pct_prest_vuln:.1f}%)**.
- El hogar típico de la muestra se concentra alrededor de **2 a 3 integrantes** (media = {media_tam:.2f}; mediana = {mediana_tam:.0f}).
- Existen diferencias importantes entre aglomerados urbanos, aunque la ausencia de `PONDERA` impide interpretar estas comparaciones como estimaciones poblacionales.
- El tamaño del hogar se asocia negativamente con el IPCF (rho = {rho:.3f}), pero la regresión lineal simple tiene **bajo poder explicativo (R² = {modelo.rvalue**2:.3f})** y presenta problemas en sus residuos.
- Un modelo explicativo más completo debería incorporar variables como educación, condición de actividad, formalidad laboral, cantidad de perceptores de ingreso, rama de actividad y dependencia económica del hogar.
'''))

## Reproducibilidad

La notebook intenta primero cargar `data/eph_hogares_t3_2025.csv`. Si el archivo no está disponible, descarga automáticamente la planilla original compartida en Google Drive y lee la hoja `Datos`.

Dependencias principales: `pandas`, `numpy`, `matplotlib`, `scipy`, `requests` y `openpyxl`.